---
title: "Configuration Management"
subtitle: "From infrastructure to running services with Ansible"
author: "Salvo Nicotra"
format:
   revealjs:
        width: 1280
        height: 800
        slide-level: 4
incremental: true
theme: white
chalkboard: true
css: style.css
smaller: true
scrollable: true
include-before: |
    <img src="images/unict-logo.png" class="custom-logo" alt="Logo">
include-after: |
      <div class="custom-footer">
        *** Cloud Systems - 2025/2026 ***
      </div>
---

# Configuration Management {background-color="black" background-image="https://otrs.com/wp-content/uploads/Cnfiguretaion_Management-Featured_Image.jpg" background-size="100%"}

## The missing part


::: {.fragment}

**Using Terraform**

We had provisioned 3 virtual machines and installed docker using cloud-init

```bash
tofu apply
# → manager VM running
# → worker1 VM running
# → worker2 VM running
# Docker installed via cloud-init
```

:::

::: {.fragment}

**Still some manual steps...**

```bash
ssh manager "docker swarm init"
TOKEN=$(ssh manager "docker swarm join-token -q worker")
ssh worker1 "docker swarm join --token $TOKEN manager:2377"
ssh worker2 "docker swarm join --token $TOKEN manager:2377"
scp docker-compose.yml manager:~
ssh manager "docker stack deploy -c docker-compose.yml cacca"
```

:::


::: {.callout-important .fragment}
Terraform manages **infrastructure** (VMs, networks, storage).  
Something else must handle **configuration** (packages, services, app deployment).
:::

## What is Configuration Management?

> *"Configuration management is a process for maintaining computer systems, servers, applications, network devices, and other IT components in a **desired, consistent state**."*  
> [Red Hat](https://www.redhat.com/en/topics/automation/what-is-configuration-management)

::::{.columns}

::: {.fragment .column width="50%"}

**What it manages**

- Operating system packages & updates
- Configuration files (`/etc/docker/daemon.json`, `nginx.conf`, …)
- System services (start, stop, enable, reload)
- Users, groups, SSH keys
- Application deployment & lifecycle

:::

::: {.fragment .column width="50%"}

**Key properties**

- **Idempotent:** applying the same configuration twice produces the same result — no side effects
- **Declarative or scripted:** describe *what* you want, or *how* to get there
- **Version-controlled:** configuration lives in Git alongside application code
- **Auditable:** every change is tracked and reproducible

:::

::::


## CM Tools Landscape

::::{.columns}

::: {.fragment .column width="50%"}

**Agent-based**

Managed nodes run a persistent **daemon** (agent) that polls a central server or reacts to push notifications.

| Tool | Model | Language |
|---|---|---|
| **[Puppet](https://www.puppet.com/)** | Pull (agent → server) | Puppet DSL / Ruby |
| **[Chef](https://www.chef.io/)** | Pull (agent → Chef Server) | Ruby DSL |
| **[SaltStack](https://saltproject.io/)** | Push + Pull hybrid | YAML + Jinja2 |

*Pros:* continuous drift detection, rich reporting  
*Cons:* agent installation, port requirements, extra attack surface

:::

::: {.fragment .column width="50%"}

**Agentless**

No daemon on managed nodes — the control node connects over **SSH** (or WinRM) and runs tasks remotely.

| Tool | Transport | Language |
|---|---|---|
| **Ansible** | SSH / WinRM | YAML playbooks |
| **[Fabric](https://www.fabfile.org/)** | SSH | Python API |
| **Terraform provisioners** | SSH / WinRM | HCL + scripts |

*Pros:* zero footprint, works on any SSH-accessible host  
*Cons:* connection overhead per run, no persistent state

:::

::::

::: {.callout-note .fragment}
There is no universally better model: large enterprises often run Puppet/Chef for fleet management, while cloud-native teams tend to prefer Ansible for its low friction.
:::


## IaC vs Configuration Management

::::{.columns}

::: {.fragment .column width="33%"}

**Infrastructure as Code**  
*(Terraform, OpenTofu, CloudFormation)*

- Provisions and destroys *resources*: VMs, VPCs, load balancers, object storage
- Manages resource *lifecycle*: create, update, destroy
- Tracks desired state in a state file or cloud service
- Typically **does not** know what runs inside a VM

:::

::: {.fragment .column width="33%"}

**Configuration Management**  
*(Ansible, Puppet, Chef, SaltStack)*

- Configures *what runs inside* existing machines
- Installs packages, writes config files, starts services, deploys applications
- Idempotent by design: converge to desired state on every run
- Operates at the **OS and application layer**

:::

::: {.fragment .column width="33%"}

**The Overlap**

- CM tools can *provision* cloud resources (Ansible AWS modules, Chef Provisioning)
- IaC tools can *configure* machines (Terraform `null_resource` + `remote-exec`)
- **[Packer](https://developer.hashicorp.com/hcp/docs/packer)** sits in between: bakes a machine image with CM tools, then IaC launches it — config becomes immutable
- **Kubernetes** adds a third layer: once the cluster exists, K8s manages workload state independently of IaC or CM

:::

::::

::: {.callout-tip .fragment}
In practice: use **IaC for the infrastructure layer** and **CM for the configuration layer** — keep them loosely coupled via inventory handoff (e.g. Terraform `outputs.tf` → Ansible `hosts.ini`).
:::


## The big picture

::::{.columns}

:::{.fragment .column width="40%"}

```{mermaid}
flowchart TD
    CI["CI/CD Pipelines"]
    A["1 — Infrastructure as Code"]
    B["2 — Configuration Management"]
    C["3 — Container Orchestration"]
    D["4 — Application Delivery"]

    CI -->|"provisioning"| A
    CI -->|"re-configure nodes"| B
    A -->|"VMs ready"| B
    B -->|"K8s cluster"| C
    C -->|"Cluster ready"| D
    CI -->|"GitOps"| D
```
:::

::: {.fragment .column width="60%"}

| Layer | Tool (this course) | Responsibility |
|---|---|---|
| CI/CD | GitHub Actions | Trigger IaC, CM, deployments |
| Infrastructure | OpenTofu | Create VMs, networking |
| Configuration | **Ansible** ← *we are here* | on premises and cloud services |
| Orchestration | Kubernetes | Schedule & manage workloads |
| App delivery | Helm / manifests | Deploy applications in K8s or Cloud Services |

:::
::::

::: {.callout-note .fragment}
This course walks through **all layers**: by the end, a single `tofu apply` + `ansible-playbook site.yml` will stand up a Kubernetes cluster ready to receive workloads.
:::


# What is Ansible? {background-color="#1A1A2E" background-image="https://docs.ansible.com/images/bull_welcome.svg" background-size="50%"} 

## Ansible 101

::::{.columns}

::: {.fragment .column width="60%"}

> *"Ansible is an open source IT automation engine that **automates provisioning, configuration management, application deployment, orchestration**, and many other IT processes."*  
> [Red Hat](https://www.redhat.com/en/topics/automation/what-is-ansible)

- Created by Michael DeHaan, 2012; acquired by Red Hat in 2015
- Written in Python; control node needs only Python + SSH
- **No agents** on managed nodes: just SSH key and Python
- Playbooks written in **YAML**: human-readable, version-controllable
- Huge ecosystem: [Ansible Galaxy](https://galaxy.ansible.com) community roles & collections

:::

::: {.fragment .column width="40%"}

```{mermaid}
flowchart TB
    C["Control Node<br/>(your laptop / CI runner)"]
    J(( ))
    M["manager"]
    W["worker1"]

    C -->|SSH| J
    J --> M
    J --> W

    style J fill:transparent,stroke:transparent
```
:::

::::

## Core Concepts

| Concept | What it is | Example |
|---|---|---|
| **Inventory** | List of hosts to manage | `hosts.ini`, dynamic from Terraform outputs |
| **Playbook** | Ordered list of *plays*; each play maps hosts → tasks | `swarm.yml` |
| **Task** | A single action executed by a module | `apt: name=docker.io state=present` |
| **Module** | A reusable unit of work (built-in or from Galaxy) | `apt`, `service`, `copy`, `community.docker.docker_swarm` |
| **Role** | Reusable, shareable bundle of tasks + defaults + files | `geerlingguy.docker` |
| **Handler** | Task triggered only when notified (e.g. restart after config change) | `restart docker` |
| **Fact** | Variable auto-collected from the managed node | `ansible_facts['os_family']` |
| **Variable** | User-defined or host/group var; override with `-e` | `docker_version: "24.0"` |

## Install Ansible

<https://docs.ansible.com/projects/ansible/latest/installation_guide/intro_installation.html>

```bash
# Control node only (Ubuntu / Debian)
sudo apt update && sudo apt install -y ansible

# Or via pip (latest version)
pip install ansible

# Verify
ansible --version
# ansible [core 2.20.1]
```

::: {.callout-note}
**Managed nodes** require nothing extra — Ansible pushes a temporary Python script over SSH, executes it, then cleans up.
:::

# LAB: Manage a cluster of multipass nodes

:::{.callout-note}
The goal of this lab is to setup a cluster of 3 nodes using "just" multipass and manage with Ansible 

Reference in the repo is lab/multipass-ansible
:::

## Key

```bash
cd lab/multipass-ansible
# Generate a dedicated key for this lab
ssh-keygen -t ed25519 -f multipass-ansible -N "" 
```

::: {.callout-note}
Use a **dedicated key** (`multipass-ansible`) rather than your personal `id_rsa` it's a good discussion point, why ?
:::

## Cloud Init

Let's use cloud-init to copy the pubkey in the vms

```bash
cat > cloud-init.yml <<EOF
#cloud-config
users:
  - name: ubuntu
    sudo: ALL=(ALL) NOPASSWD:ALL
    ssh_authorized_keys:
      - $(cat multipass-ansible.pub)
EOF
```

## Start the Cluster

```bash
multipass launch --name manager  --cpus 1 --memory 1G --cloud-init cloud-init.yml
multipass launch --name worker1  --cpus 1 --memory 1G --cloud-init cloud-init.yml
multipass launch --name worker2  --cpus 1 --memory 1G --cloud-init cloud-init.yml

multipass list
```

## Ansible inventory file

<https://docs.ansible.com/projects/ansible/latest/inventory_guide/intro_inventory.html#inventory-basics-formats-hosts-and-groups>

We are going to define 3 groups, managers and workers and a swarm cluster

```bash
MANAGER_IP=$(multipass info manager --format json | jq -r '.info.manager.ipv4[0]')
WORKER1_IP=$(multipass info worker1 --format json | jq -r '.info.worker1.ipv4[0]')
WORKER2_IP=$(multipass info worker2 --format json | jq -r '.info.worker2.ipv4[0]')
```

::::{.columns}

::: {.fragment .column width="50%"}

**Static inventory (`hosts.ini`)**

```bash
cat > hosts.ini <<EOF
[managers]
manager ansible_host=$MANAGER_IP ansible_user=ubuntu

[workers]
worker1 ansible_host=$WORKER1_IP ansible_user=ubuntu
worker2 ansible_host=$WORKER2_IP ansible_user=ubuntu

[swarm:children]
managers
workers

[swarm:vars]
ansible_ssh_private_key_file=./multipass-ansible
EOF
```

:::

::: {.fragment .column width="50%"}

**YAML equivalent (`hosts.yml`)**

```bash
cat > hosts.yml <<EOF
all:
  children:
    managers:
      hosts:
        manager:
          ansible_host: $MANAGER_IP
          ansible_user: ubuntu
    workers:
      hosts:
        worker1:
          ansible_host: $WORKER1_IP
          ansible_user: ubuntu
        worker2:
          ansible_host: $WORKER2_IP
          ansible_user: ubuntu
  vars:
    ansible_ssh_private_key_file: ./multipass-ansible
EOF
```

:::

::::

## Ad-Hoc Commands

Quick one-liners without writing a playbook
<https://docs.ansible.com/projects/ansible/latest/command_guide/index.html>

```bash
# Ping all hosts (connectivity check)
ansible -i hosts.ini all -m ping

# Gather facts from managers
ansible -i hosts.ini managers -m setup

# Run a shell command
ansible -i hosts.ini swarm -m shell -a "whoami"

# Install a package (idempotent)
ansible -i hosts.ini swarm -m apt -a "name=curl state=present" --become
```

::: {.callout-tip}
`-m` = module, `-a` = module arguments, `--become` = sudo escalation.
:::

## Anatomy of a Playbook
<https://docs.ansible.com/projects/ansible/latest/playbook_guide/index.html>

```bash
cat > install_docker.yml <<EOF
# install_docker.yml
---
- name: Install Docker on all swarm nodes       # Play name
  hosts: swarm                                  # Target group
  become: true                                  # Privilege escalation (sudo)

  tasks:

    - name: Update apt cache
      apt:
        update_cache: true
        cache_valid_time: 3600

    - name: Install docker.io
      apt:
        name: docker.io
        state: present

    - name: Ensure Docker service is started and enabled
      service:
        name: docker
        state: started
        enabled: true

    - name: Add ubuntu user to docker group
      user:
        name: ubuntu
        groups: docker
        append: true
EOF
```

## Running a Playbook

```bash
# Apply
ansible-playbook -i hosts.ini install_docker.yml

# Check what would change without applying
ansible-playbook -i hosts.ini install_docker.yml --check

# Verify Docker is installed
multipass exec manager -- docker --version

# Cleanup
multipass delete --all
multipass purge
```

::: {.callout-note}
**Idempotency in action:** run the playbook twice — the second run shows `changed=0`.
:::

## Ansible vs cloud-init

| | cloud-init | Ansible |
|---|---|---|
| **Trigger** | Once at first boot | Any time, on demand |
| **Re-runnable** | ❌ (runs once) | ✅ (idempotent) |
| **Target** | Single node (self-config) | Multiple nodes in parallel |
| **Use case** | Bootstrap before SSH is live | Any post-boot configuration |
| **Format** | YAML (`cloud-config`) | YAML playbooks |
| **Conditionals/loops** | Limited | Full Jinja2 templating |

::: {.callout-tip .fragment}
**Best practice:** use cloud-init to install base tools (Docker) and set up SSH keys; use Ansible for everything that needs to be reproducible, testable, and re-run.
:::

# Terraform + Ansible {background-color="white" background-image="https://dev-to-uploads.s3.amazonaws.com/uploads/articles/82hu1y08dv8n3262yi71.png" background-size="100%"}

## Connecting the dots

::: {.callout-warning}
Use Terraform to provision infrastructure, then Ansible to configure it

how do we pass the dynamic inventory (IPs) from Terraform to Ansible?
:::

## Combined Workflow

```{mermaid}
flowchart LR
    A["tofu apply"] -->|"provisions VMs"| B["manager,worker1,worker2"]
    A -->|"writes"| C["hosts.ini"]
    C --> D["install_docker.yml"]
    B -.->|"target"| D

    style A fill:#7b42bc,color:#fff
    style D fill:#EE0000,color:#fff
```

::: {.callout-tip .fragment}
The key insight: Terraform writes `hosts.ini` as part of `tofu apply`, so Ansible always has up-to-date IPs.
:::


## Project Structure

```
multipass-ansible-terraform/
├── terraform/
│   ├── main.tf            ← provision VMs + write inventory
│   ├── variables.tf
│   ├── outputs.tf
│   ├── cloud-init.tpl     ← template: injects SSH key at boot
│   └── inventory.tpl      ← template: generates hosts.ini
└── ansible/
    ├── hosts.ini          ← generated by Terraform ✨
    └── install_docker.yml
```

::: {.callout-note .fragment}
`hosts.ini` and `cloud-init.rendered.yml` are generated — add them to `.gitignore`.
:::


## Terraform: Provision + Write Inventory

```hcl
# main.tf 
resource "multipass_instance" "manager" {
  name           = "manager"
  image          = var.ubuntu_image
  cloudinit_file = "${path.module}/cloud-init.yml"
}

resource "multipass_instance" "worker" {
  count          = var.worker_count        # 2 by default
  name           = "worker${count.index + 1}"
  image          = var.ubuntu_image
  cloudinit_file = "${path.module}/cloud-init.yml"
}

# Option B: write hosts.ini automatically
resource "local_file" "inventory" {
  filename = "${path.module}/../ansible/hosts.ini"

  content = templatefile("${path.module}/inventory.tpl", {
    manager_ip      = multipass_instance.manager.ipv4[0]
    worker_ips      = [for w in multipass_instance.worker : w.ipv4[0]]
    ssh_private_key = var.ssh_private_key_path
  })
}
```

::: {.fragment}
`local_file` is a built-in Terraform resource — no extra provider needed.
:::


## Dynamic Inventory: `local_file` + template

Terraform writes `ansible/hosts.ini` automatically as part of `tofu apply`:

```hcl
resource "local_file" "inventory" {
  filename = "${path.module}/../ansible/hosts.ini"
  content  = templatefile("${path.module}/inventory.tpl", {
    manager_ip      = multipass_instance.manager.ipv4[0]
    worker_ips      = [for w in multipass_instance.worker : w.ipv4[0]]
    ssh_private_key = var.ssh_private_key_path
  })
}
```

```ini
# inventory.tpl → rendered hosts.ini
[managers]
manager ansible_host=${manager_ip} ansible_user=ubuntu

[workers]
%{ for i, ip in worker_ips ~}
worker${i + 1} ansible_host=${ip} ansible_user=ubuntu
%{ endfor ~}

[all:vars]
ansible_ssh_private_key_file=${ssh_private_key}
```

::: {.callout-tip .fragment}
No extra script, no manual step — the inventory is always in sync with the infrastructure.
:::


## Ansible Playbook

```yaml
# install_docker.yml
---
- name: Install Docker on all nodes
  hosts: swarm
  become: true

  tasks:

    - name: Update apt cache
      apt:
        update_cache: true
        cache_valid_time: 3600

    - name: Install docker.io
      apt:
        name: docker.io
        state: present

    - name: Ensure Docker service is started and enabled
      service:
        name: docker
        state: started
        enabled: true

    - name: Add ubuntu user to docker group
      user:
        name: ubuntu
        groups: docker
        append: true
```


## Key

```bash
cd lab/multipass-ansible-terraform/terraform

# Generate a dedicated key for this lab
ssh-keygen -t ed25519 -f id_ed25519 -N ""
```


Terraform will read `id_ed25519.pub` automatically and embed it in each VM via `cloud-init.tpl`:

```hcl
resource "local_file" "cloud_init" {
  filename = "${path.module}/cloud-init.rendered.yml"
  content  = templatefile("${path.module}/cloud-init.tpl", {
    ssh_public_key = trimspace(file("${path.module}/id_ed25519.pub"))
  })
}
```


## LAB: Terraform + Ansible

:::{.callout-note}
The goal of this lab is to provision a 3-node cluster with Terraform and configure it fully with Ansible — no manual steps.

Reference: `lab/multipass-ansible-terraform`
:::

```bash
cd lab/multipass-ansible-terraform

# 0. Generate a dedicated SSH key
ssh-keygen -t ed25519 -f terraform/id_ed25519 -N ""

# 1. Provision VMs — also writes ansible/hosts.ini automatically
cd terraform
tofu init
tofu plan
tofu apply

# 2. Verify connectivity
cd ..
ansible -i ansible/hosts.ini all -m ping

# 3. Install Docker on every node
ansible-playbook -i ansible/hosts.ini ansible/install_docker.yml

# 4. Verify Docker is installed
ansible -i ansible/hosts.ini all -m shell -a "docker --version"

# Cleanup
tofu -chdir=terraform destroy
```
